# Completez por tramos y recurrencia Gutenberg--Richter

Este notebook implementa el flujo para el catálogo sísmico regional clasificado de 260 km. Ejecuta las regresiones de completez sobre conteos acumulados por intervalos de magnitud, evalúa sus coeficientes R² y construye las tasas acumuladas con el periodo completo propio de cada umbral. El detalle del algoritmo está en `analisis_recurrencia.py`.

Intervalos usados: [3, 3.3), [3.3, 3.5), [3.5, 4), [4, 4.5), [4.5, 5), [5, 5.5), [5.5, 6), [6, 6.5), [6.5, 7) y [7, 7.5). Los límites son cerrados por la izquierda y abiertos por la derecha. Se omite el año parcial 2026.

## Flujo y columnas calculadas

1. Se valida Mw, fecha y sismofuente, y se retienen años calendario completos.
2. Para cada sismofuente y tramo, se crea la serie anual de conteos y su suma acumulada desde el primer año. Se prueban quiebres que dejan al menos 10 años y 3 eventos en cada segmento. El quiebre minimiza la suma de cuadrados de dos regresiones lineales independientes. Se reportan los R² de ambos segmentos, el R² del ajuste segmentado y la pendiente posterior. Solo se usa el inicio si hay al menos 10 eventos en el tramo y R² posterior ≥ 0.90; en caso contrario se marca no estimable/no validado.
3. Para cada umbral Mᵢ se cuentan los eventos incrementales [Mᵢ,Mᵢ₊₁) dentro de la ventana completa propia de ese umbral. En esa misma ventana se vuelven a contar todos los eventos Mw ≥ Mᵢ para obtener N_acum(Mᵢ). Así, λᵢ = N_acum(Mᵢ)/tᵢ y log10(λᵢ) se calculan con exposiciones coherentes, aunque los tᵢ varíen entre umbrales.
4. Por conjunto, se ajusta log10(λᵢ) = a − bMᵢ por mínimos cuadrados usando solo tramos con completez validada y tasa positiva. Se requieren al menos tres umbrales válidos.

El CSV de trayectorias conserva el año, el conteo anual y el acumulado que sustenta cada ajuste; las tablas de completez y recurrencia guardan las métricas y tasas.

In [ ]:
from pathlib import Path
import importlib.util
import sys

cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
script_path = next(
    (
        candidate
        for root in search_roots
        for candidate in (
            root / "codigo" / "analisis_recurrencia.py",
            root / "Entrega_02" / "codigo" / "analisis_recurrencia.py",
        )
        if candidate.is_file()
    ),
    None,
)
if script_path is None:
    raise FileNotFoundError(f"No se encontró analisis_recurrencia.py desde {cwd}")

spec = importlib.util.spec_from_file_location("analisis_recurrencia", script_path)
if spec is None or spec.loader is None:
    raise ImportError(f"No se pudo cargar el módulo de análisis: {script_path}")
analysis = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = analysis
spec.loader.exec_module(analysis)
resultados = analysis.run_analysis()

print("Completez regional:")
print(resultados["completez"].query("Conjunto == 'Catálogo regional'").to_string(index=False))
print("\nRecurrencia regional:")
print(resultados["recurrencia"].query("Conjunto == 'Catálogo regional'").to_string(index=False))
print("\nAjustes Gutenberg-Richter:")
print(resultados["ajustes"].to_string(index=False))

## Curvas de atenuación exploratorias para Neiva

El módulo `analisis_atenuacion.py` aplica la relación de Zhao et al. (2006) a cada registro y calcula la distancia epicentral al Parque Santander. Las entradas que puede modificar están al comienzo del módulo en `CONFIG` y `MODEL_PARAMETERS`; también se exponen aquí antes de ejecutar el análisis.

**Suposiciones:** el punto de análisis usa las coordenadas de referencia definidas para el Parque Santander (2.9262767, -75.2892111); suelo medio se representa con la clase III y su término de sitio de Zhao; se usa PGA por defecto (la opción `Sa(0.5 s)` está disponible aparte); el campo `Z` se interpreta como profundidad focal en km, limitada a 0–125 km; las fuentes cortical, interplaca e intraplaca se asignan respectivamente a corteza, interfaz y slab. No se cuenta con mecanismos focales, por lo que `F_R=0`. Los residuales intra e interevento se fijan en cero, así que las predicciones son medianas condicionales, no percentiles ni acelerogramas. Zhao fue calibrada con sismos de Japón; los resultados para Neiva son exploratorios y no constituyen una GMPE local validada.

La distancia media por fuente y tramo se informa como resumen, pero la curva se calcula evento por evento con magnitud, distancia y profundidad de cada registro. Los percentiles de la tabla resumen describen dispersión entre eventos predichos; no representan la desviación estándar del modelo ni una tasa de amenaza.

In [ ]:
attenuation_script = next(
    (
        candidate
        for root in search_roots
        for candidate in (
            root / "codigo" / "analisis_atenuacion.py",
            root / "Entrega_02" / "codigo" / "analisis_atenuacion.py",
        )
        if candidate.is_file()
    ),
    None,
)
if attenuation_script is None:
    raise FileNotFoundError(f"No se encontró analisis_atenuacion.py desde {cwd}")

attenuation_spec = importlib.util.spec_from_file_location(
    "analisis_atenuacion", attenuation_script
)
if attenuation_spec is None or attenuation_spec.loader is None:
    raise ImportError(f"No se pudo cargar el módulo: {attenuation_script}")
attenuation = importlib.util.module_from_spec(attenuation_spec)
sys.modules[attenuation_spec.name] = attenuation
attenuation_spec.loader.exec_module(attenuation)

# Modifique estas entradas antes de ejecutar, si el escenario lo requiere.
configuracion_atenuacion = dict(attenuation.CONFIG)
configuracion_atenuacion["site_name"] = "Neiva - Parque Santander"
configuracion_atenuacion["site_latitude"] = 2.9262767
configuracion_atenuacion["site_longitude"] = -75.2892111
configuracion_atenuacion["site_class"] = "III"
configuracion_atenuacion["intensity_measure"] = "PGA"

resultados_atenuacion = attenuation.run_analysis(
    config=configuracion_atenuacion
)
print("Resumen por sismofuente y tramo de magnitud:")
print(resultados_atenuacion["resumen"].to_string(index=False))

## Archivos de salida

- `datos/datos estadistica/acumulado_anual_por_tramo_magnitud.csv`: N anual y N acumulado por año, intervalo y conjunto.
- `datos/datos estadistica/completez_por_tramo_magnitud.csv`: años de inicio, exposición, pendientes y R² por tramo.
- `datos/datos estadistica/tabla_recurrencia_gr_por_tramo.csv`: conteos incrementales/acumulados, exposición, λ, log10(λ) y parámetros a, b, R².
- `datos/datos estadistica/ajuste_gutenberg_richter_sismofuentes.csv`: resumen del ajuste por conjunto.
- `datos/datos estadistica/conteos_incrementales_ventana_umbral.csv`: desglose de cada conteo acumulado por intervalo, en la ventana temporal del umbral.
- `datos/datos estadistica/predicciones_atenuacion_neiva_pga_por_evento.csv`: aceleración mediana condicional y distancia por evento.
- `datos/datos estadistica/resumen_atenuacion_neiva_pga_por_fuente_magnitud.csv`: resumen por sismofuente y tramo de magnitud.
- `datos/datos estadistica/parametros_atenuacion_neiva_pga.json`: parámetros y entradas utilizados para reproducir el escenario.
- `figuras/completez_acumulada_*.png`: acumulados anuales y regresiones segmentadas.
- `figuras/curvas_recurrencia_sismofuentes.png`: tasas acumuladas y rectas de Gutenberg--Richter.
- `figuras/curvas_atenuacion_neiva_pga_cortical.png`, `curvas_atenuacion_neiva_pga_interplaca.png` y `curvas_atenuacion_neiva_pga_intraplaca.png`: gráficos ampliados, uno por sismofuente.